In [1]:
import torch 
from IPython import display 
from d2l import torch as d2l

是把数字数组变成张量tensor , 张量就是能放数字的容器， 标量， 向量， 矩阵， 更高维度的数组运算， torch的核心工作就是提供tensor这种数据结构， 在tensor上做数学运算， 矩阵乘法， 广播， 自动求导， backward()就是靠它， 字符串 文本， 一般不会直接变成张量， 除非你把字符编码成数字， 那是NLP里的事 现在不用管

In [ ]:
batch_size = 256
train_iter,test_iter = d2l.load_data_fashion_mnist(batch_size)


batch_size 是没批次多少张， 不是分成多少份 ， 每批次装256张， FashionMNIST训练集中一共60000张图片， 60000/256 = 235 批 
数据集里每一张图都和它的标签配成一对，（图片，标签） 。比如， （一张裤子的图片，1） train_iter每次给你一箱，这一箱里是256张图，和256张标签， 也是配成一对给你的， （256张图，256个标签）
for a,b in zip(...) 没次拿到一对， 拆成a 和b 
for X,y in train_iter 每次拿到一对，拆成X(图片)和y(标签)
所以是的 图片和标签是分开装的，但配对给出的。
2.break 对，拿完第一箱就停，3.【256，1，28，28】怎么读， 这是4个维度， 每个数字是这个维度有多大， 不是长度
256 这一箱有256张图，1 每张图有1个颜色的通道， 灰度图，彩色图是3 ：红绿蓝
28 每张图28行
28 每张图28列
28 * 28 就是像素的高和宽
SCL array[1...256,1...1,1...28,1...28] of real 
和M3连起来， 你手算X是256 * 784 。这里的784 就是 1 * 28 * 28 拉成平行的一行， 模型里会把[256,1,28,28]变成[256,784],再去乘784*10的W
y是[256] 256个标签， 每个标签是 0-9 的类表编号， 代码里不是one-hot 是直接写编号， 上次提过 
4.为什么是 235箱， 60000 / 256 = 234.375 
前234箱每箱装满了256张， 一共是59904 张， 还剩96张， 单独装成第235箱， 最后一箱不满， 但也算一箱 ， 

1.以后采集焊接图片，也要这样分类， 每张图都必须要有一个标签， 告诉模型， 这张是合格还是砂眼， 最常见， 最简单的方式就是按文件夹 分类， 焊接数据， 合格/ 放合格的图片， 砂眼/放砂眼的图片， 过度氧化/...
pytorch 有现成的工具能直接读 这样的结构， 文件夹名就是标签， 最重要的判定标准要一致  同一种缺陷不能这次放在砂眼， 下次放在合格 
2.为什么1 * 28 * 28 能乘成784  不是把像素值乘起来， 是把同样的像素重新排一下， 像素一个都没有变， 只是摆放方式变了
原来的28 行 28 列 像一个28 * 28 的鸡蛋托盘，拉平了之后把28 行首尾相接起来， 数据一样 只是数组的形状变了， 这个操作在Pytorch 里叫reshape , 3.能不能把别的维度乘起来， 能乘起来的总数不能变， [256,1,28,28] 一共 256 * 1 * 28 * 28  = 200704 个数， 只要新的形状， 乘起来是 200704 就行，

但能做和有意义是两会事 ， 我们要的是每一行是一张图， M3里说行 = 样本， 所以第一维度256 必须被保留， 后面的3维合成784 如果把256和别的维度乘在一起，几张图就混在一行里的了形状合法但是没有意义， 

彩色图 是 256，3，28，28  这里的256张彩色的图， 拉平后就是每张图3* 28 *28 2352 个数W 就要变成 2356 * 10  以后你的焊接图片是彩色的， ，尺寸也会打的多， 比如 224 * 224 的彩色图片，拉平了就是 150528 个数， 这就是为什么后面要学CNN 它不拉平， 能保留那个像素挨着那个像素 的信息

5.y的前10个 
[0,7,8,5] 0-9一共是10种 0 T恤   1 裤子   2 套衫   3 连衣裙   4 外套5 凉鞋  6 衬衫   7 运动鞋  8 包      9 短靴


总结：
乘法 ： 本来就是在数一个排得整整齐齐的东西有多少个 
28 * 28 一张图， 28 行， 每行 28 个， 
1 * 28 * 28  一个颜色层， 每层一张 28 * 28 
256 * ...784  = 256 张图， 每张784个 
一层套一层， 乘法口诀只是背结果， 现在我看到了它在数什么 

数学和代码都是为了表达一个具体的东西， 形状变成什么样， 要看我们想让它代表什么， 这里我们要的是一行， 一张图所以， 256行，  256张图上下叠起来
纠正-1 必须正好除尽不会有不满的最后一行， 如果除不尽直接报错，reshape 不会把多出来的单独放一行， 每一行都必须是满的784 

print(X.reshape(256,784).shape)

In [3]:
for X,y in train_iter:
    print("X_shape:",X.shape,"y_shape:",y.shape)
    break 

print(len(train_iter))
print(y[:10])

X_shape: torch.Size([256, 1, 28, 28]) y_shape: torch.Size([256])
235
tensor([8, 5, 8, 1, 9, 2, 5, 0, 3, 4])


In [4]:

print(X.reshape(256,784).shape)
C = X.reshape(256,784)
print(X.shape)
print(C.shape)
#print (X.reshape(-1,784).shape)
##print (X.reshape(256,28,28).shape)
##print (X.reshape(-1,1000).reshape)
##print (X.reshape(10,784).shape)



torch.Size([256, 784])
torch.Size([256, 1, 28, 28])
torch.Size([256, 784])


torch.Size object has no attribute shape  说的是这个torch.Size 形状， 对象没有形状属性
X是原来的形状， C是变形后的

只拖μ，曲线左右平移，形状不变， 只拖σ 曲线位置不动 ， 变高变瘦或者变矮变胖， 
1.分布和均匀是什么意思 ， 就是回答一个问题， 分布， 随机取数的时候， 哪些数出现得多， 哪些数出现得少， 
均匀分布，每个数出现得机会一样， 最好的例子就是掷1 到6 每个面出现的机会都是六分之一， 没有谁更受偏爱， torch.rand 就是这样， 0 到 1 之间每个位置取到的机会一样， 
正态分布， 数出现的机会不一样， 中间多， 两边少， 就像焊点直径集中在5.0附近， torch.randn是这样， 
所以均匀说的是分布的型状， 平的， 不是一个数值 。 
2.均值，方差，标准差， 
均值不是均匀， 是平均值， 

In [5]:
num_inputs = 784 
num_outputs = 10 

W = torch.normal(0,0.01, size = (num_inputs,num_outputs),requires_grad=True)
b = torch.zeros(num_outputs,requires_grad= True)

num_inputs = 784 每张图拉平后784个像素 模型输入， num_outputs = 10 10个类别的分数模型输出
模型做的事就是吃进 784个数 吐出 10个分数， 所以叫inputs, outputs 
torch.normal（0,0.01 这里你要记住，因为你遇到了好几次了 均值为0 标准差为0.01 意思是 造一堆在0 的附近， 很小的数， 大多是在 -0.03 到0.03之间的数 
为什么要随机小数 ， W 总的从某个地方开始， M3里说过W一开始是随机的，靠训练调， 用很小的数一开始酸楚的分数都很接近0 相当于瞎猜 ， 这就是手算的loss = 0.69 那个状态
size = 784 ,10 就是形状， 括号里写的行和列， 
requires_grad = True 告诉pythorch 这是要调的参数， 帮我记着参与了哪些计算， 这样调用backward() 时能算出它的梯度， 存在w.grad 里， 
梯度还是导数， 导数是一个参数的坡度， w 里有7840个数， 每个都有自己的坡度，这个7840 个坡度和在一起叫梯度， 所以， w.grad 和w 形状一样， 也是784*10 
b 为什么长度是num_outputs M3 里学过b是每个类别的基础分， 有几个类别就有几个， 所以是10 一开始都设置成0 同样requires_grad=True , 训练时也要调， 位置，= W，b ,这一格造出来就是7840 + 10个旋钮，后面训练就是在拧它， 
新建一个


你说这两种说法在数学上竟然是等价的， 
1.先说标准的正态分布，均值0 标准差1 里取一个数，再乘0.01 
2.直接从均值0 标准差0.01的正态分布里取一个数
两种方式得到的数， 分布完全一样， 所以你以后再别的代码里看到的， torch.randn(784,10)*0.01 和你写的torch.normal(0,0.01,.... )效果相同， 只是写法不一样， 
W = torch.normal(0,0.01,size = (784,10),requires_grad=True) 翻译成人话就是， 生成784*10=7840个随机数，它们们的中心来自一个中心是0 ，标准差是0.01的正态分布 
按上面的规律来说的话， 
约68%的数载-0.01 到 0.01之间 
约99.7%的数载-0.03 到 0.03之间
也就是说W一开始全是非常接近0的小数 ， 有正有负，但没有两个完全一样的 
昨天说的， 乘0.01是这么来的， torch.randn生成的是μ=0、σ=1 的标准的正态分布， 99.7%在-3 到 3 之间 ， 把它们全部都乘0.01 范围就缩小在-0.03 - 0.03之间， 西格玛也就变成了0.01 就像把一个钟形的曲线横向压扁了100倍， 所以torch.randn(10000) 从标准的正态分布取1万个数， 
print('均值'，x.mean()) #猜：接近多少？


In [6]:
import torch 
x = torch.randn(10000)  #从标准的正态分布中取1万个数
print ("均值：",x.mean())  #猜接近多少 ？ 
print ("标准差：",x.std()) #猜接近多少 ？ 

#有多少比例落在 -1 和 1之间 
print("±1σ 内的比例:",((x>-1) & (x < 1)).float().mean())

## 这里的逐个判断，是否在-1 到 1 之间， .float()把True变成了1.0 false变成了0.0 
#[False, True, True, True, False] 就变成了 [0.0, 1.0, 1.0, 1.0, 0.0] 
#.mean 就是求平均， （0+1+1+1+0）/5 =0.6 这个小技巧在以后算准确率的时候会一直用， 
#判断对不对 变成0和1 求平均，就是正确率

#有多少比例落在 -3 和3之间 ？ 
print("±3σ 内的比例:",((x>-3) & (x < 3)).float().mean())

#把它们乘 0.01 之后
y=x * 0.01 
print("乘0.01后的标准差：", y.std())
print("乘0.01后的最大值，最小值：",y.max(),y.min())

均值： tensor(0.0186)
标准差： tensor(0.9955)
±1σ 内的比例: tensor(0.6900)
±3σ 内的比例: tensor(0.9975)
乘0.01后的标准差： tensor(0.0100)
乘0.01后的最大值，最小值： tensor(0.0410) tensor(-0.0372)


In [7]:
torch.randn(1000) * 2 + 3 

tensor([ 3.8844,  4.0041,  2.1536,  4.3350,  5.5550,  5.4317,  5.7655,  3.1777,
         4.6485,  1.8333,  3.5895,  4.1499,  1.1067,  5.3152,  3.6407,  3.0525,
         3.7345,  3.8225,  2.5157,  4.4836,  1.3625,  3.7615,  5.8493,  3.8459,
         3.9102,  1.1734,  4.5566,  2.5849,  6.1099, -1.3340,  1.4027,  3.5488,
         0.6719,  0.3592,  0.1722,  3.6814,  1.6713,  1.4830,  4.2828,  8.0544,
        -0.0285,  6.7918,  0.6189,  1.8314,  3.3064,  0.2960,  0.3425,  6.8432,
         2.8419,  4.6779,  4.6263,  2.6786,  4.8610,  1.3453,  2.7161,  2.6391,
         2.0389,  3.9295,  2.2133,  3.7730,  1.3857,  2.9064,  3.2120,  1.2914,
         3.2025,  0.7282,  2.5332,  1.6446,  1.6196,  2.7513,  1.2972,  4.4399,
         4.9530,  2.7436,  3.0153,  2.8064,  2.5195,  5.2346, -0.7542,  5.5539,
        -0.1149,  4.1024,  3.3935,  4.5806,  0.9922,  3.4668,  6.5658,  2.9211,
         2.2992,  3.1014,  2.5285,  6.2161, -0.8326,  0.9602,  1.3554,  2.8950,
         4.2316,  2.7361, -0.1181,  3.64

In [8]:
torch.tensor([4.,5.,6.]).std(unbiased=False)
#会得到1.0  

tensor(0.8165)

In [ ]:
import torch 
X_demo=torch.rand(1,784)  #假装是一张图，784个像素，值在0-1。改名 X_demo，避免覆盖真正训练用的 X
print("X_demo:",X_demo)
for S in [0.01,1.0]:    #这里for循环的 把列表里的值一个个拿出来，放进S， 每次执行一遍下面的缩进代码
    W_demo = torch.normal(0,S,size=(784,10))  #改名 W_demo，这里没写 requires_grad=True，如果叫 W 会把真正模型的权重覆盖掉，而且还会丢了梯度
    o = X_demo @ W_demo #X_demo：1行784列 一张假图， W_demo784 行10列， 初始权重。 这里的o是分数不是概率， 也不是权重， 每个数对应一个类， 分数越高， 模型越倾向于这一类， 分数可以数负数， 加起来也不等于 1 
    P = torch.softmax(o,dim = 1)

    print(f"标准差{S}:")
    print(" 10个分数:", o.round(decimals=2))
    print(" 10个概率:", P.round(decimals=3))
    print(" 最大概率:", P.max().item())

@就是矩阵乘法， 等于torch.matmul 
torch.matmul(X,w)+b 
X @ w + b 是一模一样的 

问题一，学会如何查说明书
2.你以前常见的for i in range(3). 其实时同一个结构， 只是range(3) 产生的是 0，1，2 这里直接给了一个列表而已。
3.两个标准差所以循环两次， 但是它和训练时的呼唤不是一回事， 这里只是把同意个实验用两种设置各做了一遍

我这里把3件事揉在一起，softmax , 交叉熵损失， 标准差， 这很正常，因为它们都有平方， 开方， e ,ln ,这些操作 ， 
1.softmax把分数变成概率，不是平方， 先对每个分数取e的次方， 再除以总和， 
为什么用e^x 全是正数，加起来是1 ，而且分数高的概率也高
为什么用e的次方， 不用平方，但是平方有一个致命的问题， 会打乱顺序，分数-3，平方是9 ，分数2 ， 平方是4，结果是-3，记过-3 反而比2大，这就错了，e的次方永远都是正数，而且分数越大顺序结果不会乱
以e为底， 分数做指数，即，它永远大于0 
2.交叉熵损失， 看正确答案，的概率有多高
假设正确答案的类别是A，概率是0.665 损失= -ln(0.665)=0.41 
正确的答案的概率越接近1，损失越接近0 ， 概率越小， 损失越大，  

这里我们来复习一下交叉熵， 为什么是-ln(p) 第一步我们想要一个什么样的惩罚， 模型看一张图片， 给出正确的概率是p,我们想要一个惩罚分数， 满足， 1.p = 1 (完全猜对， 非常确定) -> 惩罚， = 0 
2.p 越小， 越不信正确答案， 惩罚越大， 
3.惩罚不能是负数 
 第二步： 最简单的想法， 1-p ， 行不行？ 
 p = 0.9 惩罚 0.1 ；p = 0.1 惩罚 0.9 。 好想也满足条件， 
 

In [10]:
import torch
s = torch.tensor([2.,1.,0.])
p = torch.softmax(s,dim=0)
print("方法一：",-torch.log(p[0]))
print("方法二：",-s[0] + torch.log(torch.exp(s).sum()))
print("ln(e^2) =",torch.log(torch.exp(torch.tensor(2.))))

方法一： tensor(0.4076)
方法二： tensor(0.4076)
ln(e^2) = tensor(2.)


In [ ]:
X_demo = torch.tensor([[1,2,3]]) #形状（1，3），只是演示矩阵乘法，不要用 W 这个名字，避免覆盖真正的模型权重
W_demo = torch.tensor([[1,0],[0,1],[1,1]])  #形状（3，2）
X_demo @ W_demo 
print(X_demo @ W_demo)

In [12]:
X = torch.tensor([[1.0,2.0,3.0],[4.0,5.0,6.0]])
X.sum(0,keepdim=True),X.sum(1,keepdim=True)

(tensor([[5., 7., 9.]]),
 tensor([[ 6.],
         [15.]]))

In [13]:
def softmax(X):
    X_exp= torch.exp(X)
    partition = X_exp.sum(1,keepdim=True)
    return X_exp / partition # 这里应用了广播机制


In [14]:
X = torch.normal(0,1,(2,5))
X_prob = softmax(X)
X_prob,X_prob.sum(1)

(tensor([[0.0916, 0.0494, 0.2090, 0.1952, 0.4547],
         [0.2514, 0.0967, 0.4176, 0.0538, 0.1805]]),
 tensor([1., 1.]))

In [15]:
#定义模型
def net(X):
    return softmax(torch.matmul(X.reshape((-1,W.shape[0])), W)+b)

In [16]:
y = torch.tensor([0,2])
y_hat = torch.tensor([[0.1,0.3,0.6],[0.3,0.2,0.5]])
y_hat [[0,1],y]

tensor([0.1000, 0.5000])

In [17]:
def cross_entropy(y_hat,y):
    return - torch.log(y_hat[range(len(y_hat)),y])

cross_entropy(y_hat,y)

tensor([2.3026, 0.6931])

In [18]:
def accuracy(y_hat,y):
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1 :
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())

In [19]:
accuracy(y_hat,y)/len(y) 

0.5

In [20]:
def evaluate_accuracy(net,data_iter):
    """计算在指定数据集上的模型的精度"""
    if isinstance(net, torch.nn.Module):
        net.eval() #将模型设置为评估模式
    metric = Accumulator(2) 
    with torch.no_grad():
        for X,y in data_iter:
            metric.add(accuracy(net(X),y),y.numel())
    return metric[0] / metric[1]

In [21]:
class Accumulator:
    """在n个变量上累加"""
    def __init__(self,n):
        self.data = [0.0] * n 

    def add(self,*args):
        self.data = [a + float(b) for a,b in zip(self.data,args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self,idx):
        return self.data[idx]

In [22]:
evaluate_accuracy(net,test_iter)

RuntimeError: shape '[-1, 3]' is invalid for input of size 200704

In [ ]:
#训练
def train_epoch_ch3(net,train_iter, loss , updater): #@save
    """训练模型一个迭代周期 (定义见第3章)"""
    #将模型设置为训练模式
    if isinstance(net,torch.nn.Module):
        net.train()
    #训练损失总和，训练准确度总和， 样本数
    metric = Accumulator(3)
    for X,y in train_iter:
        #计算梯度并更新参数
        y_hat= net(X)
        l = loss(y_hat,y)
        if isinstance(updater,torch.optim.Optimizer):
            #使用Pytorch内置的优化器和损失函数
            updater.zero_grad()
            l.mean().backward()
            updater.step()
        else:
            #使用定制的优化器和损失函数
            l.sum().backward()
            updater(X.shape[0])
        metric.add(float(l.sum()),accuracy(y_hat,y),y.numel())
    #返回训练损失函数和训练精度
    return metric[0]/metric[2],metric[1]/metric[2]

In [ ]:
class Animator: #@save
    """ 在动画中绘制数据 """
    def __init__(self,xlabel=None,ylabel=None,legend= None,xlim=None,
                 ylim=None, xscale='linear',yscale='linear',
                 fmts=('-','m--','g-.','r:'),nrows=1,ncols=1,
                 figsize=(3.5,2.5)):
        #增量地绘制多线条
        if legend is None:
            legend = []
        d2l.use_svg_display()
        self.fig,self.axes = d2l.plt.subplots(nrows,ncols,figsize=figsize)
        if nrows*ncols ==1:
            self.axes = [self.axes, ]
        #使用lambda函数捕获参数
        self.config_axes = lambda: d2l.set_axes(
        self.axes[0],xlabel,ylabel,xlim,ylim,xscale,yscale,legend)
        self.X,self.Y,self.fmts = None,None,fmts

    def add(self,x,y):
        #向图表中添加多个数据点
        if not hasattr(y,"__len__"):
            y = [y]
        n = len(y)
        if not hasattr(x,"__len__"):
            x = [x]* n
        if not self.X:
            self.X = [[] for _ in range(n)]
        if not self.Y:
            self.Y = [[] for _ in range(n)]
        for i , (a,b) in enumerate(zip(x,y)):
            if a is not None and b is not None:
                self.X[i].append(a)
                self.Y[i].append(b)
        self.axes[0].cla()
        for x, y, fmt in zip(self.X ,self.Y, self.fmts):
            self.axes[0].plot(x,y,fmt)
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)

In [ ]:
def train_ch3(net,train_iter,test_iter,loss,num_epochs,updater):#@save 
    """训练模型（定义见第3章）"""
    animator = Animator(xlabel='epoch',xlim=[1,num_epochs],ylim = [0.3,0.9],
                       legend = ['train loss','train acc','test acc'])
    
    for epoch in range(num_epochs):
        train_metrics = train_epoch_ch3(net,train_iter,loss,updater)
        test_acc = evaluate_accuracy(net,test_iter)
        animator.add(epoch + 1, train_metrics + (test_acc,))
    train_loss, train_acc = train_metrics
    print(f"最后一轮：train_loss={train_loss:.3f}, train_acc={train_acc:.3f}, test_acc={test_acc:.3f}")
    assert train_loss < 0.5, train_loss
    assert train_acc <= 1 and train_acc > 0.7, train_acc 
    assert test_acc <= 1 and test_acc > 0.7 ,test_acc 
        

In [ ]:
lr = 0.1 

def updater(batch_size):
    return d2l.sgd([W,b],lr,batch_size)

In [ ]:
num_epochs = 10 
train_ch3(net,train_iter,test_iter,cross_entropy,num_epochs, updater)

In [ ]:
def predict_ch3(net,test_iter,n=6): #@save
    """预测标签(定义见第3章)"""
    for X,y in test_iter:
        break
    trues = d2l.get_fashion_mnist_labels(y)
    preds = d2l.get_fashion_mnist_labels(net(X).argmax(axis=1))
    titles = [true + '\n' + pred for true, pred in zip(trues, preds)]
    d2l.show_images(
        X[0:n].reshape((n,28,28)),1,n,titles = titles[0:n])

predict_ch3(net,test_iter)